# Multi-input Deep Learning for News Classification

**Author:** Enrico Maineri  
**Course:** Machine Learning, Artificial Neural Networks and Deep Learning � Part 2  
**Exam session:** 16 June 2026

This notebook implements a reproducible multi-label news classifier using the supplied text, Bag-of-Words vectors, and publication metadata. Its sections follow assignment points 1�5 and document preprocessing, architecture, training, and held-out evaluation.


## Setup

In [ ]:
import numpy as np, tensorflow as tf, random, string, zipfile, pickle as pk
from collections import Counter

SEED = 213
def set_seed(s=SEED):
    np.random.seed(s); tf.random.set_seed(s); random.seed(s)
set_seed()
print('TF version:', tf.__version__)

## Data download


In [ ]:
!wget -q https://frasca.di.unimi.it/MLDNN/input_data.zip -O data.zip
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('unzipped_data')
with open('unzipped_data/input_data.pkl', 'rb') as f:
    data = pk.load(f)

print('keys in archive:', list(data.keys()))

texts  = data['texts']      # list: raw text of each news (headline + short description)
years  = data['years']      # list: publication year
months = data['months']     # list: publication month
Y = np.array(data['labels']).astype('float32')  # ndarray [N, K], multi-hot targets

import scipy.sparse as sp
_known = {'texts', 'years', 'months', 'labels'}
bow_key = [k for k in data.keys() if k not in _known][0]
bow_raw = data[bow_key]
if not sp.issparse(bow_raw):
    bow_raw = np.asarray(bow_raw, dtype='float32')
print('BoW key:', bow_key, '| BoW shape:', bow_raw.shape, '| sparse:', sp.issparse(bow_raw))

N, K = Y.shape
assert N == len(texts) == len(years) == len(months), "Input fields must have the same number of samples."
assert Y.ndim == 2 and np.isin(Y, [0, 1]).all(), "Labels must be a binary multi-hot matrix."
assert K == 18, f"Expected 18 categories, found {K}."
assert bow_raw.shape[0] == N, "BoW rows must align with news entries."
labels_per_item = Y.sum(axis=1)
positive_per_label = Y.sum(axis=0)
assert labels_per_item.min() == 2 and labels_per_item.max() == 3, "Expected 2-3 labels per news item."
assert positive_per_label.min() >= 1000 and positive_per_label.max() <= 4000, "Label support is outside the assignment range."
print(f'N = {N} news, K = {K} categories')
print(f'Labels per item: min={labels_per_item.min()}, max={labels_per_item.max()}, '
      f'mean={labels_per_item.mean():.2f}')
print(f'Positive samples per label: min={positive_per_label.min()}, max={positive_per_label.max()}')
print('example text:', texts[0][:100], '...')


# 1. MODEL CHOICE

This is a **multi-label classification** task: each news item can belong to more than one of the 18 semantic categories. The model has three input branches:

- an Embedding and LSTM branch for variable-length text;
- a dense branch for the supplied Bag-of-Words (BoW) count vector;
- a metadata branch for publication year and month.

The branch representations are concatenated and passed to a shared classifier. The output has 18 independent sigmoid probabilities, one for each category, so multiple labels can be predicted for one article.


# 2. INPUT - preprocessing and model input

The archive supplies article text, a BoW count matrix, publication year and month, and a multi-hot target matrix. The pipeline prepares three model inputs:

1. **Text:** clean and tokenize each article, map tokens to a training-only vocabulary, then pad or truncate each sequence to 100 integer IDs.
2. **BoW:** remove terms occurring in fewer than three training documents and standardize retained count features using statistics fitted on training rows.
3. **Metadata:** one-hot encode year and month using encoders fitted on training rows.

The dataset is split before fitting any vocabulary or transformer. Validation and test rows are transformed using training-fitted preprocessing only. The labels already have the required multi-hot representation.


### 2.0 - Train / Validation / Test split (70 / 15 / 15)
We hold out a test set (touched only once, at the very end) and a validation set (used for early stopping / model selection).

We use a reproducible random 70/15/15 split. Standard single-label stratification does not directly apply because each sample has multiple labels. The code reports per-label prevalence in each split so the distribution can be checked. Validation supports model selection and early stopping; test is reserved for final evaluation.


In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(N)
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=SEED)
idx_val,   idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=SEED)
print(f'train={len(idx_train)}  val={len(idx_val)}  test={len(idx_test)}')

# Compare positive-label prevalence across the three splits.
for split_name, split_idx in (("train", idx_train), ("validation", idx_val), ("test", idx_test)):
    prevalence = Y[split_idx].mean(axis=0)
    print(f"{split_name:10s}: mean labels/sample={Y[split_idx].sum(axis=1).mean():.2f}; "
          f"per-label prevalence range={prevalence.min():.3f}�{prevalence.max():.3f}")


### 2.1 - Text cleaning
The logical steps that turn a raw text into word tokens: replace `--` with a space, split on whitespace, **remove punctuation**, keep only **alphabetic** tokens, **lowercase**.

In [ ]:
def clean_text(t):
    t = t.replace('--', ' ')
    table = str.maketrans('', '', string.punctuation)
    words = [w.translate(table) for w in t.split()]
    words = [w.lower() for w in words if w.isalpha()]
    return words

docs = [clean_text(t) for t in texts]
print('cleaned example:', docs[0][:15])

### 2.2 - Vocabulary for the text branch (training set only)

The vocabulary is built from training documents only. It retains words appearing in at least MIN_DF distinct training documents and is capped at MAX_VOCAB entries. Index 0 is reserved for padding and out-of-vocabulary words. Validation and test text do not influence vocabulary fitting.


In [ ]:
MIN_DF, MAX_VOCAB = 2, 10000
doc_freq, term_freq = Counter(), Counter()
for i in idx_train:
    for w in set(docs[i]): doc_freq[w]  += 1
    for w in docs[i]:      term_freq[w] += 1

kept = [w for w in doc_freq if doc_freq[w] >= MIN_DF]
kept = sorted(kept, key=lambda w: term_freq[w], reverse=True)[:MAX_VOCAB]
word2idx = {w: i + 1 for i, w in enumerate(kept)}   # 0 reserved for padding/OOV
V = len(word2idx)
print(f'vocabulary size V = {V}')

### 2.3 - Text branch input: integer sequences + padding
Each news becomes a sequence of word indices, **padded/truncated** to `MAXLEN = 100` (lengths range 1-100 in the data). Final shape `[N, 100]`, integer domain `{0, ..., V}` (0 = padding).

In [ ]:
MAXLEN = 100
seqs  = [[word2idx[w] for w in d if w in word2idx] for d in docs]
X_seq = tf.keras.utils.pad_sequences(seqs, maxlen=MAXLEN, padding='post', truncating='post')
print('X_seq shape:', X_seq.shape, '| dtype:', X_seq.dtype)

### 2.4 - BoW branch input: frequency reduction and standardization

The archive provides a 10,000-dimensional Bag-of-Words count vector for each item. We retain terms appearing in at least MIN_DF_BOW training documents, then fit StandardScaler on training rows and apply that transformation to every split. The reduced matrix is densified for the neural-network input; this step can require substantial memory for larger datasets.


In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

MIN_DF_BOW = 3   # drop words occurring in fewer than 3 training documents (frequency-based reduction)

# document frequency per word, computed on the TRAINING rows only
bow_train = bow_raw[idx_train]
if sp.issparse(bow_train):
    df_bow = np.asarray((bow_train > 0).sum(axis=0)).ravel()
else:
    df_bow = (bow_train > 0).sum(axis=0)

keep_cols = np.where(df_bow >= MIN_DF_BOW)[0]
X_bow = bow_raw[:, keep_cols]
X_bow = X_bow.toarray() if sp.issparse(X_bow) else np.asarray(X_bow)
X_bow = X_bow.astype('float32')

scaler = StandardScaler().fit(X_bow[idx_train])
X_bow = scaler.transform(X_bow).astype('float32')
print('BoW reduced shape:', X_bow.shape, f'(kept {len(keep_cols)} of {bow_raw.shape[1]} words)')

### 2.5 - Metadata branch input: one-hot encoding

Year and month are categorical values, not continuous measurements. One-hot encoding represents each value as its own binary feature without implying a numerical distance between category IDs. Encoders are fitted only on training rows; an unseen value is safely represented by all-zero columns for that field.


In [ ]:
from sklearn.preprocessing import OneHotEncoder
yr = np.array(years).reshape(-1, 1)
mo = np.array(months).reshape(-1, 1)
ohe_year  = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(yr[idx_train])
ohe_month = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(mo[idx_train])
X_meta = np.concatenate([ohe_year.transform(yr), ohe_month.transform(mo)], axis=1).astype('float32')
print('X_meta shape:', X_meta.shape)   # expected [N, 23]

### 2.6 - Assemble the three inputs per split
The model receives a **list of three inputs** `[sequence, bow, metadata]`, with value domains: sequence in `{0,...,V}`, BoW standardized (real), metadata in `{0,1}`.

In [ ]:
X_train = [X_seq[idx_train], X_bow[idx_train], X_meta[idx_train]]
X_val   = [X_seq[idx_val],   X_bow[idx_val],   X_meta[idx_val]]
X_test  = [X_seq[idx_test],  X_bow[idx_test],  X_meta[idx_test]]
Y_train, Y_val, Y_test = Y[idx_train], Y[idx_val], Y[idx_test]
print('shapes:', [a.shape for a in X_train], Y_train.shape)

# 3. OUTPUT and LOSS

The model outputs 18 independent probabilities using a sigmoid activation. Each estimates whether its category applies to the article. Because categories are not mutually exclusive, softmax would impose an unsuitable sum-to-one constraint.

The target matrix contains one binary value per sample and category. Binary cross-entropy is applied independently to each output and averaged across the batch and categories.


For a target vector \(y \in \{0,1\}^K\) and predicted probabilities \(p_k\), the per-sample binary cross-entropy is

\[
L(y,p) = -\frac{1}{K} \sum_{k=1}^{K} \left[y_k \log(p_k) + (1-y_k) \log(1-p_k)\right].
\]

Keras binary_crossentropy implements this independent-label objective for the sigmoid outputs.


# 4. MODEL CONFIGURATION

The following sections show the model pipeline, hyperparameter search, and final training. Configurations are compared using validation macro-F1; validation loss controls early stopping during final training. The held-out test set is not used for model choices.


### 4.a - Model composition (pipeline)

Each input is processed by its own branch. The resulting vectors are concatenated and passed through a dense classification head.

~~~~
  Text [B,100]             BoW [B,D_BOW]          Metadata [B,23]
       |                         |                       |
  Embedding [B,100,E]      Dense(128)+BN+ReLU        identity
       |                         |                       |
  LSTM [B,256]              [B,128]                   [B,23]
       \_________________________|_______________________/
                                 |
                     Concatenate [B,407]
                                 |
                     Dense(256)+ReLU -> Dropout
                     Dense(64)+ReLU  -> Dropout
                     Dense(18)+Sigmoid
~~~~

D_BOW is the number of features remaining after training-set frequency filtering. The concatenated width is 407: 256 LSTM features, 128 BoW features, and 23 metadata features.


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, LSTM, Dense, Dropout,
                                     BatchNormalization, Activation, Concatenate)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import HeNormal

def build_model(emb_dim, hidden_sizes, dropout_rate, l2_reg):
    # Text branch: mask padding so it does not affect the LSTM state.
    in_seq = Input(shape=(MAXLEN,), name='text')
    x_t = Embedding(input_dim=V + 1, output_dim=emb_dim, mask_zero=True)(in_seq)
    x_t = LSTM(256)(x_t)   # LSTM hidden size fixed (not tuned)
    # BoW branch: project standardized counts to a compact representation.
    in_bow = Input(shape=(X_bow.shape[1],), name='bow')
    x_b = Dense(128, kernel_regularizer=l2(l2_reg), kernel_initializer=HeNormal())(in_bow)
    x_b = BatchNormalization()(x_b)
    x_b = Activation('relu')(x_b)
    # Metadata branch: concatenate the one-hot year and month features directly.
    in_meta = Input(shape=(X_meta.shape[1],), name='meta')
    # concatenation + dense head (number & size of hidden layers = hidden_sizes)
    c = Concatenate()([x_t, x_b, in_meta])
    for h in hidden_sizes:
        c = Dense(h, activation='relu', kernel_regularizer=l2(l2_reg), kernel_initializer=HeNormal())(c)
        c = Dropout(dropout_rate)(c)
    out = Dense(K, activation='sigmoid')(c)
    return Model(inputs=[in_seq, in_bow, in_meta], outputs=out)

set_seed()
demo = build_model(emb_dim=64, hidden_sizes=(256, 64), dropout_rate=0.5, l2_reg=1e-4)
demo.summary()


**Model architecture figure:** the following Matplotlib diagram shows the three input branches and their fusion.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Draw the architecture with Matplotlib so the figure does not depend on Graphviz.
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(0, 13); ax.set_ylim(0, 5); ax.axis("off")

def box(x, y, w, h, label, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                           linewidth=1.2, edgecolor="#334155", facecolor=color)
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9)

box(0.2, 3.7, 2.0, 0.8, "Text sequence\n[batch, 100]", "#dbeafe")
box(2.7, 3.7, 2.0, 0.8, "Embedding + LSTM\n[batch, 256]", "#bfdbfe")
box(0.2, 2.1, 2.0, 0.8, "BoW counts\n[batch, D_BOW]", "#dcfce7")
box(2.7, 2.1, 2.0, 0.8, "Dense + BN + ReLU\n[batch, 128]", "#bbf7d0")
box(0.2, 0.5, 2.0, 0.8, "Year + month\n[batch, 23]", "#fef3c7")
box(5.5, 2.0, 2.0, 1.0, "Concatenate\n[batch, 407]", "#e9d5ff")
box(8.0, 2.0, 2.0, 1.0, "Dense head\n256 -> 64", "#ddd6fe")
box(10.5, 2.0, 2.0, 1.0, "18 sigmoid outputs\n[multi-label]", "#fbcfe8")
for y in (4.1, 2.5):
    ax.annotate("", xy=(5.5, 2.5), xytext=(4.7, y), arrowprops=dict(arrowstyle="->", color="#475569", lw=1.3))
ax.annotate("", xy=(5.5, 2.5), xytext=(2.2, 0.9), arrowprops=dict(arrowstyle="->", color="#475569", lw=1.3))
for y in (4.1, 2.5):
    ax.annotate("", xy=(2.7, y), xytext=(2.2, y), arrowprops=dict(arrowstyle="->", color="#475569", lw=1.3))
ax.annotate("", xy=(8.0, 2.5), xytext=(7.5, 2.5), arrowprops=dict(arrowstyle="->", color="#475569", lw=1.3))
ax.annotate("", xy=(10.5, 2.5), xytext=(10.0, 2.5), arrowprops=dict(arrowstyle="->", color="#475569", lw=1.3))
ax.set_title("Multi-input neural network for multi-label news classification", fontsize=12)
plt.tight_layout()
plt.show()


### 4.b - Optimization and model selection

The model uses Adam with learning rate 1e-3, gradient clipping, He initialization for dense layers, L2 regularization, dropout, and early stopping. Hyperparameters are compared on validation macro-F1. To keep the search manageable, it varies embedding dimension, dense-head widths, and dropout across eight configurations while holding other settings fixed.

The search trains on a reproducibly sampled subset of training rows for a small number of epochs. The selected configuration is then trained on the full training split with validation-based early stopping. The test split is reserved for the final evaluation.


In [ ]:
import itertools
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import f1_score

# Vary the main capacity and regularization choices while keeping the search small.
param_grid = {
    'emb_dim':      [32, 64],                  # tuned (embedding dimension)
    'hidden_sizes': [(256, 64), (128, 64)],    # tuned (size of MLP hidden layers)
    'dropout_rate': [0.3, 0.5],                # tuned (dropout rate)
    'l2_reg':       [1e-4],
    'batch_size':   [256],
    'patience':     [2],
    'clipvalue':    [1.0],
}

SUBSET, EPOCHS_SEARCH = 3000, 5   # subset + few epochs to stay within the time budget
# Sample training rows rather than assuming the first rows are representative.
rng = np.random.default_rng(SEED)
subset_idx = rng.choice(len(Y_train), size=min(SUBSET, len(Y_train)), replace=False)
Xs = [a[subset_idx] for a in X_train]
Ys = Y_train[subset_idx]

keys = list(param_grid.keys())
best_f1, best_cfg = -1.0, None
for values in itertools.product(*[param_grid[k] for k in keys]):
    cfg = dict(zip(keys, values))
    tf.keras.backend.clear_session(); set_seed()
    m = build_model(cfg['emb_dim'], cfg['hidden_sizes'], cfg['dropout_rate'], cfg['l2_reg'])
    m.compile(optimizer=Adam(learning_rate=1e-3, clipvalue=cfg['clipvalue']), loss='binary_crossentropy')
    m.fit(Xs, Ys, epochs=EPOCHS_SEARCH, batch_size=cfg['batch_size'], verbose=0)
    f1 = f1_score(Y_val, (m.predict(X_val, verbose=0) >= 0.5).astype(int), average='macro', zero_division=0)
    print(f"emb_dim={cfg['emb_dim']}, hidden_sizes={cfg['hidden_sizes']}, dropout={cfg['dropout_rate']} -> val macro-F1={f1:.4f}")
    if f1 > best_f1: best_f1, best_cfg = f1, cfg

print('Best config:', best_cfg, '| val macro-F1:', round(best_f1, 4))


**Final training.** The best configuration is retrained on the full training set with early stopping; this `model` is the one evaluated on the test set in point 5.

In [ ]:
tf.keras.backend.clear_session(); set_seed()
model = build_model(best_cfg['emb_dim'], best_cfg['hidden_sizes'],
                    best_cfg['dropout_rate'], best_cfg['l2_reg'])
model.compile(optimizer=Adam(learning_rate=1e-3, clipvalue=best_cfg['clipvalue']),
              loss='binary_crossentropy', metrics=['binary_accuracy'])
early = EarlyStopping(monitor='val_loss', patience=best_cfg['patience'], restore_best_weights=True)

EPOCHS = 10
history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                    epochs=EPOCHS, batch_size=best_cfg['batch_size'], callbacks=[early], verbose=2)

**Learning curves** (training vs validation loss)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Learning curve'); plt.show()

# 5. MODEL EVALUATION

The final model is evaluated on the held-out test split, which was not used for preprocessing, hyperparameter selection, or early stopping. Predictions are converted to binary labels using a fixed threshold of 0.5.

We report micro-F1 and macro-F1. Micro-F1 pools decisions across categories and is influenced more by frequent categories. Macro-F1 computes F1 per category and averages the scores, giving every category equal weight. Together they describe overall performance and performance across both frequent and less frequent categories.


The decision threshold is fixed at 0.5 for this evaluation. A different threshold could be selected using validation data in a separate experiment, but it must not be chosen by inspecting test results.


In [ ]:
from sklearn.metrics import f1_score

Y_prob = model.predict(X_test, batch_size=512, verbose=0)
Y_pred = (Y_prob >= 0.5).astype(int)

f1_micro = f1_score(Y_test, Y_pred, average='micro', zero_division=0)
f1_macro = f1_score(Y_test, Y_pred, average='macro', zero_division=0)
print(f'Test F1 (micro): {f1_micro:.4f}')
print(f'Test F1 (macro): {f1_macro:.4f}')

## Conclusion

This notebook provides an end-to-end baseline for multi-label news classification. It combines sequential text features, supplied BoW counts, and publication metadata; fits preprocessing on training data only; compares a small set of configurations on validation data; and reports micro- and macro-F1 on held-out test data.

The metrics describe this split and configuration. They are not a claim of production-level performance or a substitute for evaluation across different splits.
